# 🌀 Phase 3: Conditional DDPM — Photo → Sketch (from scratch)

**Build DDPM/DDIM diffusion from scratch in raw PyTorch.**
No HuggingFace, no diffusers — understand every line.

### How it works
- **Training:** Add noise to sketches, train U-Net to predict the noise (conditioned on photo)
- **Inference:** Start from pure noise, denoise in 50 DDIM steps guided by the photo
- **No discriminator needed** — pure MSE, no adversarial instability

### Prerequisites
- **GPU:** T4 x2 or P100 — enable GPU + Internet
- **Dataset:** face2sketch Kaggle Dataset (already uploaded)


In [ ]:
# @title 1. Setup — clone, install, verify
import os, sys

REPO = 'https://github.com/weseegod/face2sketch.git'
BASE = '/kaggle/working/face2sketch'
os.environ['FACE2SKETCH_BASE'] = BASE

if not os.path.exists(BASE):
    !git clone {REPO} {BASE}
os.chdir(BASE)
!git pull origin main

!pip install -q torch torchvision tqdm pillow

import torch
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
print(f"GPUs: {torch.cuda.device_count()}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

!python -c "from src.diffusion import ConditionalUNet, NoiseScheduler, DiffusionModel; print('✅ All imports OK')"

In [ ]:
# @title 2. Copy Data from Kaggle Dataset
import os, shutil

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

# Find data folders
data_root = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dataset' in dirs and 'test' in dirs:
        data_root = root
        break

if data_root:
    print(f'📂 Found data at: {data_root}')
    os.makedirs(f'{BASE}/data', exist_ok=True)
    for sub in ['dataset', 'test']:
        src = os.path.join(data_root, sub)
        dst = f'{BASE}/data/{sub}'
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
        n = len(os.listdir(os.path.join(dst, 'photos'))) if os.path.exists(dst) else 0
        print(f'    data/{sub}/  →  {n} photos')
else:
    print('❌ dataset/test folders not found')

def count(d):
    p = os.path.join(BASE, d, 'photos')
    return len(os.listdir(p)) // 2 if os.path.isdir(p) else 0

ds = count('data/dataset')
print(f'\n✅ Training pairs: {ds}')
assert ds > 0, '❌ No training data — cannot proceed'

In [ ]:
# @title 3. Phase 3 — Train DDPM (~8-10 hours)
import os

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

assert torch.cuda.is_available(), "⚠️ Enable GPU"
assert os.path.isdir('data/dataset'), "❌ Missing dataset"

print("🌀 Phase 3 — Conditional DDPM from Scratch")
print(f"    T=1000  |  cosine schedule  |  DDIM 50-step sampling")
print(f"    LR=2e-4  |  batch=8  |  epochs=500\n")

gpu_count = torch.cuda.device_count()
if gpu_count > 1:
    print(f"🚀 {gpu_count} GPUs detected — DataParallel enabled\n")

!python src/train_diffusion.py \
    --device cuda \
    --epochs 500 \
    --batch-size 8 \
    --lr 2e-4 \
    --patience 0 \
    --val-every 25 \
    --name phase3_v1_

print("\n✅ Training complete → checkpoints/phase3_v1_best.pt")

In [ ]:
# @title 4. Evaluate — Test Set + Sample Grid
import os, glob

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

ckpt = 'checkpoints/phase3_v1_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/phase3_v1_*.pt'))
    for c in candidates:
        if os.path.getsize(c) > 100000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('❌ No Phase 3 checkpoint found.')
else:
    print(f'📊 Evaluating {ckpt}...')
    print(f'   Size: {os.path.getsize(ckpt)/1e6:.0f}MB\n')

    # Quick visual: 8 samples
    import torch
    from src.diffusion import DiffusionModel
    from src.evaluate import evaluate_diffusion
    from src.data_loader import DATASET_MEAN, DATASET_STD

    device = torch.device('cuda')
    model = DiffusionModel(T=1000, base_ch=64, time_dim=256, device=device)
    ckpt_data = torch.load(ckpt, map_location=device, weights_only=False)
    state = ckpt_data.get('model', ckpt_data)
    if any(k.startswith('module.') for k in state.keys()):
        state = {k.replace('module.', ''): v for k, v in state.items()}
    model.model.load_state_dict(state)
    model.model.eval()
    print(f'   Epoch: {ckpt_data.get("epoch", "?")}  |  Val loss: {ckpt_data.get("val_loss", "?"):.6f}')

    # Evaluate on test set + save progress.png from training
    evaluate_diffusion(model, 'data/test', device, num_steps=50)

    # Show training progress if available
    import glob
    progress_files = sorted(glob.glob('outputs/progress.png'))
    if progress_files:
        from IPython.display import Image, display
        print('\n📸 Latest training progress:')
        display(Image(filename=progress_files[-1]))

print('\n✅ Phase 3 evaluation complete')